In [1]:
# imports

# to get my token from .env
from dotenv import load_dotenv

# get token
import os

# handle requests 
import requests 

# db adapter
import psycopg2

# insert data into staging tables
from psycopg2.extras import execute_values

In [2]:
# get token from .env

# load .env
load_dotenv()

# get token
TOKEN = os.getenv("GITHUB_TOKEN")

# get DB name
DB_NAME = os.getenv("DB_NAME")

# get DB password
PASSWORD = os.getenv("PASSWORD")

# ensure env vars exists
print("TOKEN exists:", TOKEN is not None)
print("DB_NAME exists:", DB_NAME is not None)
print("PASSWORD exists:", PASSWORD is not None)

TOKEN exists: True
DB_NAME exists: True
PASSWORD exists: True


In [3]:
# declare headers with TOKEN

headers = {
    "Authorization": f"Bearer {TOKEN}"
}

In [4]:
# get data with endpoint and headers
def get_data(endpoint, headers):

    # get data using endpoint
    url = "https://api.github.com" + endpoint
    r = requests.get(url, headers=headers)

    # (repo commit endpoint error)
    # if 409 status code then commit history not accessible
    # so do nothing
    if r.status_code == 409:
        return []
    # check for other errors 
    else:
        r.raise_for_status()

        # return json data
        return r.json()

# flatten dictionaries in repo
def flatten_repo_dictionaries(repo):

    # get the current repo dictionary 
    # return empty dict as back up
    owner = repo.get("owner") or {}
    permissions = repo.get("permissions") or {} 
    license = repo.get("license") or {}

    # keep only owner id
    repo["owner_id"] = owner.get("id")

    # flatten repo dicts
    repo["permissions_admin"] = permissions.get("admin")
    repo["permissions_maintain"] = permissions.get("maintain")
    repo["permissions_push"] = permissions.get("push")
    repo["permissions_triage"] = permissions.get("traige")
    repo["permissions_pull"] = permissions.get("pull")

    repo["license_key"] = license.get("key")
    repo["license_name"] = license.get("name")
    repo["license_spdx_id"] = license.get("spdx_id")
    repo["license_url"] = license.get("url")
    repo["license_node_id"] = license.get("node_id")
    
    repo.pop("owner", None)
    repo.pop("permissions", None)
    repo.pop("license", None)


In [5]:
try:
    # pipeline vars
    # get users after id 0
    since = 0
    # number of pages to ingest
    pages = 1

    # pagination size
    users_per_page = 10
    repos_per_page = 15
    commits_per_page = 1
    issues_per_page = 1
    
    # store user/repos/commits/issues data
    all_users_data = []
    all_repos_data = []
    all_commits_data = []
    all_issues_data = []

    # for each page get x users
    for i in range(pages):

        # get users data
        endpoint = f"/users?since={since}&per_page={users_per_page}"

        # returns list
        users = get_data(endpoint, headers)

        # break if users is empty list > stop pagination loop
        if not users:
            break

        # get the usernames to get more data
        logins = [user["login"] for user in users]

        # for each user get more detailed data
        for login in logins:
            
            # endpoint for specific user data using username
            endpoint = f"/users/{login}"

            # returns a dict per user
            user_data = get_data(endpoint, headers)

            # append user dict to all data
            all_users_data.append(user_data)

            # repo section
            
            # endpoint for specific user repos data using username
            endpoint = f"/users/{login}/repos?per_page={repos_per_page}"

            # returns a list
            repo_data = get_data(endpoint, headers)

            # flatten repo dictionaries
            for repo in repo_data:
                flatten_repo_dictionaries(repo)

            # extend repos data list to include repo data
            all_repos_data.extend(repo_data)
            
            # for each repo get the commits/issues for that repo
            for repo in repo_data:

                # get repo name for url
                repo_name = repo["name"]

                # endpoint for getting commits of a user's specific repo
                endpoint = f"/repos/{login}/{repo_name}/commits?per_page={commits_per_page}"

                # returns a list
                commit_data = get_data(endpoint, headers)

                all_commits_data.extend(commit_data)

                # this section is to ingest issues data
                
                # if repo has issues then call
                if repo["has_issues"] == True:
                    
                    # # endpoint for getting issues of a user's specific repo
                    endpoint = f"/repos/{login}/{repo_name}/issues?per_page={issues_per_page}"
    
                    # returns list
                    issue_data = get_data(endpoint, headers)
    
                    # add issue data to all issues data
                    all_issues_data.extend(issue_data)
        
        # GitHub REST api with users endpoint uses 'since' to get the users where the id is after 'since'
        # therefore, get the last id of the current call to get the next page
        since = users[-1]["id"]

# handle HTTP status errors first, then catch other request failures
except requests.exceptions.HTTPError as e:
    print("HTTP error:", e)
    
except requests.exceptions.RequestException as e:
    print("Other request error:", e)

In [6]:
# data size
print("Number of users:", len(all_users_data))
print("Number of repos:",len(all_repos_data))
print("Number of commits:", len(all_commits_data))
print("Number of issues:", len(all_issues_data))

Number of users: 10
Number of repos: 143
Number of commits: 143
Number of issues: 32


In [7]:
# connect to postgres DB
#conn = psycopg2.connect(f"dbname={DB_NAME} user=postgres password={PASSWORD}")

In [8]:
# Open cursor for db operations
#cur = conn.cursor()

#conn.rollback()

In [9]:
# query to get table names for insert 
sql_query = """
     SELECT column_name
     FROM information_schema.columns
     WHERE table_name = 'users'
     ORDER BY ordinal_position
 """

# cur.execute(sql_query)

# table_columns = [row[0] for row in cur.fetchall()]

In [10]:
# query to insert users into DB table
# sql_query = f"""
# INSERT INTO users ({",".join(table_columns)})
# VALUES %s
# """

# get dict keys for list of columns
# dict_columns = list(all_users_data[0].keys())

# convert data into list of tuples for insertion
# user_data = [
#     tuple(user.get(col) for col in dict_columns)
#     for user in all_users_data
# ]

# execute_values(cur, sql_query, user_data)

In [11]:
#conn.commit()
#cur.close()
#conn.close()

In [12]:
for x in list(all_repos_data[0].keys()):

    print(x, f"\t{type(all_repos_data[0][x])}")

id 	<class 'int'>
node_id 	<class 'str'>
name 	<class 'str'>
full_name 	<class 'str'>
private 	<class 'bool'>
html_url 	<class 'str'>
description 	<class 'NoneType'>
fork 	<class 'bool'>
url 	<class 'str'>
forks_url 	<class 'str'>
keys_url 	<class 'str'>
collaborators_url 	<class 'str'>
teams_url 	<class 'str'>
hooks_url 	<class 'str'>
issue_events_url 	<class 'str'>
events_url 	<class 'str'>
assignees_url 	<class 'str'>
branches_url 	<class 'str'>
tags_url 	<class 'str'>
blobs_url 	<class 'str'>
git_tags_url 	<class 'str'>
git_refs_url 	<class 'str'>
trees_url 	<class 'str'>
statuses_url 	<class 'str'>
languages_url 	<class 'str'>
stargazers_url 	<class 'str'>
contributors_url 	<class 'str'>
subscribers_url 	<class 'str'>
subscription_url 	<class 'str'>
commits_url 	<class 'str'>
git_commits_url 	<class 'str'>
comments_url 	<class 'str'>
issue_comment_url 	<class 'str'>
contents_url 	<class 'str'>
compare_url 	<class 'str'>
merges_url 	<class 'str'>
archive_url 	<class 'str'>
download

In [ ]:
all_repos_data[0]["topics"]

In [ ]:
# pop permissions and license : /